# PatchFinders — Saved Model Evaluation Notebook

This notebook evaluates saved model weights across **In-Distribution** and **Out-of-Distribution (OOD)** test datasets, displays performance metrics tables, plots mAP comparisons, and runs inference demonstrations.

### Step 1: Import Dependencies & Project Modules

In [4]:
%pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 4.5 MB/s  0:00:0027.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.6/837.6 kB 780.7 kB/s  0:00:005 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 609.0 kB/s  0:01:37 eta 0:00:010:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [ultralytics]m 10/11 [ultralytics]s-runtime-32]

[notice] A new release of pip is available: 26.0 -> 26.1.2
[notice] To update, run: /Users/edwinsjohn/Documents/patch_finders/venv/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

# Import modular CLI & evaluation handlers
from src.cli.evaluate import evaluate, OOD_SETS
from src.visualization.plot_results import plot_ood_comparison
from src.cli.predict import predict

### Step 2: Locate Saved Model Weights

In [9]:
# Auto-detect available trained weights or fallback to pretrained models
weight_candidates = [
    "runs/detect/runs/train/yolov8s_baseline-4/weights/best.pt",
    "runs/detect/runs/train/yolov8s_baseline/weights/best.pt",
    "runs/detect/runs/train/yolov8n_baseline-4/weights/best.pt",
    "runs/train/yolov8m_baseline/weights/best.pt",
    "weights/yolov8s.pt",
    "weights/yolov8n.pt"
]

selected_weights = None
for w in weight_candidates:
    if os.path.exists(w):
        selected_weights = w
        break

if not selected_weights:
    selected_weights = "weights/yolov8s.pt"

print(f"Target Saved Weights for Evaluation: {selected_weights}")

Target Saved Weights for Evaluation: runs/detect/runs/train/yolov8s_baseline-4/weights/best.pt


### Step 3: Run Model Evaluation across Test Sets

In [ ]:
# Execute evaluation pipeline
results = evaluate(weights=selected_weights)

   PatchFinders - OOD Evaluation

[EVAL] Running: in_distribution
Ultralytics 8.4.100 🚀 Python-3.14.3 torch-2.12.1 CPU (Apple M1)
Model summary (fused): 73 layers, 11,127,906 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 95.2±66.3 MB/s, size: 70.8 KB)
val: Scanning /Users/edwinsjohn/Downloads/Eldhose_patch_finder/data/processed/val/labels.cache... 3931 images, 1225 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3931/3931 7.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 0% ──────────── 1/246 59.0s/it 17.7s<4:00:53

### Step 4: Display Summary Metrics Table

In [ ]:
metrics_csv = "outputs/metrics/ood_summary.csv"
if os.path.exists(metrics_csv):
    df_metrics = pd.read_csv(metrics_csv, index_col=0)
    print("\n================ Evaluation Summary Table ================")
    display(df_metrics)
else:
    print(f"[INFO] Metrics file not yet generated at {metrics_csv}")

### Step 5: Visualize Performance Comparison Bar Chart

In [ ]:
if os.path.exists(metrics_csv):
    plot_ood_comparison(metrics_csv)
else:
    print("No evaluation metrics CSV found to plot.")

### Step 6: Sample Inference Demonstration

In [ ]:
test_images_dir = "data/processed/test/in_distribution/images"
if os.path.exists(test_images_dir):
    print(f"Running sample inference on: {test_images_dir}")
    predictions = predict(weights=selected_weights, source=test_images_dir, conf=0.25)
else:
    print(f"[INFO] Sample images directory {test_images_dir} not found. Specify custom image path to test.")